# The query interface — feature demo

One cell per feature, on the WaterTAP seawater-RO model.

**Start the server** (from the repo root, wait for the first solve to ingest data):

```bash
uv run acquirium server --config deployments/WATERTAP/models/seawater-ro/acquirium.toml
```

**The interface in one paragraph** — `acq.query()` builds a pattern out of three
verbs (`entity`, `related`, `measurement`). Every attribute (medium, unit,
process, ...) is one vocabulary shared by filtering (`where`), projection
(`include`) and faceting (`options`/`facets`); `Not()` negates a value.
Multi-hop traversal runs as a client-side BFS, so SPARQL never sees
join-explosive chains, and `nearest=` keeps the closest match. Attribute
predicates are hidden from generic traversal by default, and aliases default to
the class name you typed.

In [1]:
from acquirium import Acquirium                     # constructor now waits for /health
from acquirium.Client.explore import Not, hidden_predicates

acq = Acquirium(server_url="localhost", server_port=8000)

## Build patterns

`entity(cls)` — class as URI or free text (server-resolved); the alias
defaults to what you typed.

In [2]:
acq.query().entity("pump").metadata()

pump
str
"""wbs:P2"""
"""wbs:intake"""
"""wbs:P1"""


`alias()` names the current node; `uri=` pins an instance (CURIEs work).

In [3]:
acq.query().entity(uri="wbs:RO").alias("ro").metadata()

ro
str
"""wbs:RO"""


`related(cls)` finds related entities — by default the *nearest* match
within 3 hops of any non-hidden predicate.

In [4]:
acq.query().entity("pump").related("tank",nearest=False,direction="downstream").metadata()

pump,tank
str,str
"""wbs:intake""","""wbs:ferric-chloride-addition"""
"""wbs:intake""","""wbs:chlorination"""
"""wbs:P2""","""wbs:storage-tank-2"""
"""wbs:intake""","""wbs:static-mixer"""
"""wbs:P1""","""wbs:storage-tank-2"""


`via=` restricts traversal to a predicate (repeatable up to
`max_depth`), a list of predicates, or `"any"`; `nearest=False` returns all
matches, `max_depth=0` opts into unbounded.

In [5]:
(acq.query().entity("System")
 .related("Equipment", via="hasMember/connectedTo", nearest=False)
 .metadata())

System,Equipment
str,str
"""wbs:pretreatment-system""","""wbs:ferric-chloride-addition"""
"""wbs:desalination-system""","""wbs:PXR"""
"""wbs:seawater-ro-plant""","""wbs:backwash-handling"""
"""wbs:posttreatment-system""","""wbs:co2-addition"""
"""wbs:seawater-ro-plant""","""wbs:storage-tank-1"""
…,…
"""wbs:posttreatment-system""","""wbs:storage-tank-2"""
"""wbs:posttreatment-system""","""wbs:lime-addition"""
"""wbs:seawater-ro-plant""","""wbs:storage-tank-3"""


`direction="upstream"/"downstream"` walks the s223 piping topology; the
step patterns each direction infers are inspectable constants in
`explore.directions` and can be passed to `via=` for nearest searches.

In [6]:
(acq.query().entity(uri="wbs:RO")
 .related("pump", direction="upstream")
 .metadata())

0,pump
str,str
"""wbs:RO""","""wbs:P2"""
"""wbs:RO""","""wbs:P1"""


## Measurements

`measurement()` attaches data-bearing points — the source's own plus its
connection points' (`include_connection_points=False` for own only);
keyword attributes filter inline, `Not()` excludes, lists mean OR.

In [7]:
from acquirium.Client.explore import Not
(acq.query().entity(uri="wbs:RO").alias("ro")
 .measurement(alias="feed").where(quantity_kind="mass flow rate", medium=Not("brine"))
 .metadata())

ro,feed
str,str
"""wbs:RO""","""wbs:RO-out-flow-mass-tds"""
"""wbs:RO""","""wbs:RO-in-flow-mass-water"""
"""wbs:RO""","""wbs:RO-out-flow-mass-water"""
"""wbs:RO""","""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO""","""wbs:RO-in-flow-mass-tds"""


In [8]:
acq.query().entity("pump").related("Pressure Exchanger").measurement(frm="*").metadata().head(10)

pump,Pressure Exchanger,pump_data,Pressure Exchanger_data
str,str,str,str
"""wbs:P2""","""wbs:PXR""",null,"""wbs:PXR-efficiency"""
"""wbs:P1""","""wbs:PXR""","""wbs:P1-efficiency""",null
"""wbs:P2""","""wbs:PXR""",null,"""wbs:PXR-brine-out-tds-concentr…"
"""wbs:P1""","""wbs:PXR""",null,"""wbs:PXR-brine-out-tds-concentr…"
"""wbs:P1""","""wbs:PXR""","""wbs:P1-mechanical-power""",null
"""wbs:P2""","""wbs:PXR""",null,"""wbs:PXR-brine-out-flow-mass-wa…"
"""wbs:P1""","""wbs:PXR""",null,"""wbs:PXR-efficiency"""
"""wbs:P1""","""wbs:PXR""",null,"""wbs:PXR-brine-out-pressure"""
"""wbs:P2""","""wbs:PXR""",null,"""wbs:PXR-brine-out-flow-mass-td…"


On an empty query, `measurement()` is the root form: every registered
stream in the plant (`frm="*"` / `frm=["a", "b"]` attach per-entity).

In [9]:
acq.query().measurement(quantity_kind="pressure").metadata()

data
str
"""wbs:RO-in-pressure"""
"""wbs:P1-out-pressure"""
"""wbs:conn-cartridge-filtration-…"
"""wbs:RO-out-pressure"""
"""wbs:PXR-brine-out-pressure"""
"""wbs:RO-out-retentate-pressure"""


`measurement(direction=..., nearest=True)` finds the closest up/downstream
measurement matching the filters.

In [10]:
(acq.query().entity(uri="wbs:P1").alias("p1")
 .measurement(direction="downstream", nearest=True, quantity_kind="pressure")
 .options("quantity_kind"))

quantity_kind,count
str,i64
"""qudtqk:Pressure""",1


## Filter, project, shape

`where()` filters any node by alias (`target=`), same attribute vocabulary
everywhere.

In [11]:
## process is its own resolver kind, so equipment classes never outrank it
(acq.query().entity("Equipment").where(process="reverse osmosis")
 .metadata())

Equipment
str
"""wbs:RO"""


`include()` adds `alias.attr` columns (placed right after their node's
column); `required=True` drops rows lacking the attribute.

In [12]:
(acq.query().entity(uri="wbs:RO").include("process").measurement(alias="m")
 .include("quantity_kind", "unit")
 .metadata())

0,0.process,m,m.quantity_kind,m.unit
str,str,str,str,str
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-membrane-area""","""qudtqk:Area""","""unit:M2"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-in-flow-mass-water""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-out-retentate-flow-mass…","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-out-retentate-flow-mass…","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-out-pressure""","""qudtqk:Pressure""",null
…,…,…,…,…
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-in-flow-mass-tds""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-out-flow-mass-tds""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO""","""nawi:Process-ReverseOsmosis""","""wbs:RO-in-temperature""","""qudtqk:Temperature""",null


`drop()` keeps a node in the pattern but out of the output (rows
deduplicate accordingly); `refocus()` moves the pointer back to an alias.

In [13]:
(acq.query().entity(uri="wbs:pretreatment-system").drop()
 .related("equipment").measurement(alias="sensor")
 .metadata())

equipment,sensor
str,str
"""wbs:intake""","""wbs:intake-in-tss-concentratio…"
"""wbs:intake""","""wbs:intake-in-toc-concentratio…"
"""wbs:cartridge-filtration""","""wbs:cartridge-filtration-out-t…"
"""wbs:intake""","""wbs:intake-in-tds-concentratio…"
"""wbs:intake""","""wbs:intake-in-flow-rate"""


`with_columns()` merges the two: plain specs include, `"-"`-prefixed
specs drop; dotted `"alias.attr"` targets any node's attribute.

In [14]:
(acq.query().entity(uri="wbs:RO").alias("ro")
 .measurement(alias="m")
 .with_columns("m.quantity_kind", "m.unit", "-ro")
 .metadata())

m,m.quantity_kind,m.unit
str,str,str
"""wbs:RO-in-temperature""","""qudtqk:Temperature""",null
"""wbs:RO-out-retentate-pressure""","""qudtqk:Pressure""",null
"""wbs:RO-out-retentate-flow-mass…","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO-out-flow-mass-tds""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO-in-pressure""","""qudtqk:Pressure""",null
…,…,…
"""wbs:RO-in-flow-mass-water""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""
"""wbs:RO-membrane-area""","""qudtqk:Area""","""unit:M2"""
"""wbs:RO-out-flow-mass-water""","""qudtqk:MassFlowRate""","""unit:KiloGM-PER-SEC"""


`include()` and `drop()` are inverses — each accepts the other's
vocabulary, so any column decision can be reversed later in the chain
(here: un-drop `ro`, un-include `unit`).

In [15]:
q = (acq.query().entity(uri="wbs:RO").alias("ro").drop()
     .measurement(alias="m").include("unit"))
q.include("ro").drop("unit").metadata()

ro,m
str,str
"""wbs:RO""","""wbs:RO-out-flow-mass-tds"""
"""wbs:RO""","""wbs:RO-out-retentate-flow-mass…"
"""wbs:RO""","""wbs:RO-in-pressure"""
"""wbs:RO""","""wbs:RO-out-retentate-pressure"""
"""wbs:RO""","""wbs:RO-in-flow-mass-water"""
…,…
"""wbs:RO""","""wbs:RO-in-temperature"""
"""wbs:RO""","""wbs:RO-in-flow-mass-tds"""
"""wbs:RO""","""wbs:RO-membrane-area"""


## Faceted exploration

`options(attr)` — the distinct values of one attribute across the current
matches, counted client-side.

In [16]:
acq.query().measurement().options("quantity_kind")

quantity_kind,count
str,i64
"""qudtqk:MassFlowRate""",10
"""qudtqk:Pressure""",6
"""qudtqk:MassConcentration""",5
"""qudtqk:Efficiency""",3
"""qudtqk:Power""",2
"""qudtqk:Temperature""",2
"""qudtqk:VolumeFlowRate""",2
"""qudtqk:Area""",1
"""qudtqk:Density""",1


`facets()` — every applicable attribute at once, falling back to
model-wide then ontology vocabulary when the pattern is empty.

In [17]:
acq.query().entity(uri="wbs:RO").alias("RO").measurement().facets()

FacetSummary('RO_data')
  type [matched]: s223:QuantifiableObservableProperty (11), ns1:VirtualPoint (11)
  medium [matched]: s223:Fluid-Water (3), nawi:Water-Seawater (2), nawi:Water-Brine (1)
  substance [matched]: nawi:Constituent-Salt (3)
  quantity_kind [matched]: qudtqk:MassFlowRate (6), qudtqk:Pressure (3), qudtqk:Area (1), qudtqk:Temperature (1)
  unit [matched]: unit:KiloGM-PER-SEC (6), unit:M2 (1)
  enumeration_kind: (no values)
  data_source: (no values)

## Data

`data()` returns the lazy DataObject; `dataframe(shape="wide")` puts
`time` first and value columns in alphabetical order; `convert_to` resolves
the target unit *jointly with the source* so only convertible matches win.

In [18]:
d = (acq.query().measurement(alias="tds", quantity_kind="mass concentration")
     .data(cast_value="float"))
d.dataframe(shape="wide").tail(3)

time,tds__wbs:cartridge-filtration-out-toc-concentration,tds__wbs:intake-in-tds-concentration,tds__wbs:intake-in-toc-concentration,tds__wbs:PXR-brine-out-tds-concentration,tds__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 21:25:53.449064 UTC,0.00213,37.009983,0.003327,62.420952,0.241329
2026-08-05 21:26:08.452869 UTC,0.002784,36.481189,0.00435,61.956656,0.235662
2026-08-05 21:26:25.509154 UTC,0.002105,36.155202,0.003288,62.711411,0.239167


In [19]:
d.convert_to("mg/L").dataframe(shape="wide").tail(3)

time,tds__wbs:cartridge-filtration-out-toc-concentration,tds__wbs:intake-in-tds-concentration,tds__wbs:intake-in-toc-concentration,tds__wbs:PXR-brine-out-tds-concentration,tds__wbs:storage-tank-3-out-tds-concentration
"datetime[μs, UTC]",f64,f64,f64,f64,f64
2026-08-05 21:25:53.449064 UTC,2.12984,37009.982899,3.327146,62420.952073,241.328931
2026-08-05 21:26:08.452869 UTC,2.784422,36481.189469,4.349706,61956.656368,235.661631
2026-08-05 21:26:25.509154 UTC,2.104581,36155.202399,3.2877,62711.410538,239.167278


## Guard rails

Attribute predicates, `subClassOf`, `hasProperty`, and `s223:cnx` are hidden
from `via="any"` traversal by default (`hide()`/`unhide()` adjust); every
attribute-taking method documents the attribute table in its docstring
(`help(q.where)`); and inspect any query with `to_sparql()` before running.

In [20]:
sorted(hidden_predicates())

['http://data.ashrae.org/standard223#cnx',
 'http://data.ashrae.org/standard223#hasConnectionPoint',
 'http://data.ashrae.org/standard223#hasMedium',
 'http://data.ashrae.org/standard223#hasProperty',
 'http://data.ashrae.org/standard223#ofMedium',
 'http://data.ashrae.org/standard223#ofSubstance',
 'http://qudt.org/schema/qudt/hasEnumerationKind',
 'http://qudt.org/schema/qudt/hasQuantityKind',
 'http://qudt.org/schema/qudt/hasUnit',
 'http://www.w3.org/1999/02/22-rdf-syntax-ns#type',
 'http://www.w3.org/2000/01/rdf-schema#subClassOf',
 'https://brickschema.org/schema/Brick/ref#hasExternalReference',
 'urn:acquirium#dataSource',
 'urn:nawi-water-ontology#hasProcess']

In [21]:
print(acq.query().entity("pump").measurement(alias="m").to_sparql())

SELECT DISTINCT ?v0 ?v1 ?ext1 ?unit1 ?extunit1
WHERE {
  ?v0 <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> ?v0_typ .
  { SELECT DISTINCT ?v0_typ WHERE { ?v0_typ <http://www.w3.org/2000/01/rdf-schema#subClassOf>* <http://data.ashrae.org/standard223#Pump> . } }
  { ?v0 ?p_e0_1 ?v1 . } UNION { ?v0 <http://data.ashrae.org/standard223#hasConnectionPoint> ?cp_e0_k1 . ?cp_e0_k1 ?p_e0_1 ?v1 . }
  ?v1 <https://brickschema.org/schema/Brick/ref#hasExternalReference> ?ext1 .
  OPTIONAL { ?v1 <http://qudt.org/schema/qudt/hasUnit> ?unit1 . }
  OPTIONAL { ?ext1 <http://qudt.org/schema/qudt/hasUnit> ?extunit1 . }
}
